In [ ]:
import os, sys, requests, pprint
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, count, max, min, avg, sum, mean, stddev, variance

In [2]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .getOrCreate()

In [39]:

# df = spark.read.parquet("C:/Marco Conti/Projetos/DIMAS/data/dados_dimas_clima.parquet")
df = spark.read.parquet("C:/Marco Conti/Projetos/DIMAS/data/dados_dimas_saude.parquet")
# df = spark.read.parquet("C:/Marco Conti/Projetos/DIMAS/data/dados_dimas_socio.parquet")
# df = spark.read.parquet("C:/Marco Conti/Projetos/DIMAS/data/tb_indicadores_clima_saude.parquet")

df = df.withColumnRenamed("Código do Município", "codigo_municipio") \
       .withColumnRenamed("Mês", "mes") \
       .withColumnRenamed("Ano", "ano") \
       .withColumnRenamed("Indicador", "indicador") \
       .withColumnRenamed("Valor", "valor") \
       .withColumnRenamed("Unidade", "unidade")


# df.printSchema()
# df.show(truncate = False )
# df.filter("indicador like 'Temperatura%' and codigo_municipio in ('110001','110003') ") \
#   .orderBy("ano", "mes", "indicador") \
#   .show(truncate = False )

# df.select("indicador").distinct().orderBy("indicador").show(truncate = False ) # 13 indicadores

# df.select("codigo_municipio").distinct().count() # 5571 municipios

# df.select("ano", "mes").groupBy("ano", "mes").count().orderBy("ano", "mes").show(truncate = False )
# # 2022 1..12 e 2023 1..8, cada grupo tem 72349 registros

In [42]:
df.filter("codigo_municipio = 312980 and indicador = 'Casos de dengue (por 100mil hab)'").show(truncate = False )

+----------------+----+---+--------------------------------+-----------+------------------+------------+-----------------+
|codigo_municipio|ano |mes|indicador                       |População  |valor             |Chave       |__index_level_0__|
+----------------+----+---+--------------------------------+-----------+------------------+------------+-----------------+
|312980          |2022|1  |Casos de dengue (por 100mil hab)|Geral      |1.1727660273137208|31298020221 |1185408          |
|312980          |2022|1  |Casos de dengue (por 100mil hab)|Idosa (60+)|0.0               |31298020221 |1185409          |
|312980          |2022|2  |Casos de dengue (por 100mil hab)|Geral      |0.5863830136568604|31298020222 |1185422          |
|312980          |2022|2  |Casos de dengue (por 100mil hab)|Idosa (60+)|0.0               |31298020222 |1185423          |
|312980          |2022|3  |Casos de dengue (por 100mil hab)|Geral      |1.1727660273137208|31298020223 |1185436          |
|312980         

In [37]:
df.createOrReplaceTempView("temp_clima")

query = \
    """Select indicador
             ,min(ano) as min_ano
             ,max(ano) as max_ano
             ,count(*) as count
         from temp_clima
        where codigo_municipio = 355280
        group by indicador

    """

spark.sql(query).show(truncate = False )



+----------------------------------------------------+-------+-------+-----+
|indicador                                           |min_ano|max_ano|count|
+----------------------------------------------------+-------+-------+-----+
|Umidade média (%)                                   |2022   |2024   |36   |
|Poluição do ar - PM₂.₅ (µg/m3)                      |2022   |2024   |36   |
|Temperatura média (°C)                              |2022   |2024   |36   |
|Poluição do ar - NO₂ (ppb)                          |2022   |2024   |36   |
|Poluição do ar - O₃ (ppb)                           |2022   |2024   |36   |
|Poluição do ar - CO (ppb)                           |2022   |2024   |36   |
|Poluição do ar - SO₂ (µg/m3)                        |2022   |2024   |36   |
|Precipitação total (mm)                             |2022   |2024   |36   |
|Temperatura extrema baixa (°C abaixo do percentil 5)|2022   |2024   |36   |
|Temperatura extrema alta (°C acima do percentil 95) |2022   |2024   |36   |